In [4]:
import sys
sys.path.insert(1, '/Users/amjonz/Documents/GitHub/mesher/src')
import mesher as msh
import gmsh
import numpy as np
import pyvista as pv
from collections import defaultdict

In [5]:
# Dimention and ID tracker, takes in surface gmsh element tags e.g. (dimention, ID's) tuples 
# and their names and tracks changes passed from fragmentation operations or other boolean mesh operations

class DimensionIDTracker:

    ''' Initializes dictionary'''
    def __init__(self):
        self.data = {}  # Stores {(dim, id): name}
        self.parent_child_map = defaultdict(list)  # Stores parent-child relationships
    
    def add_entry(self, dim_ids, names):
        """
        Adds multiple entries to the tracker.
        :param dim_ids: List of tuples [(dimension, id), ...]
        :param names: List of strings [name, ...]
        """
        if len(dim_ids) != len(names):
            raise ValueError("Dimension ID list and names list must be of the same length.")
        
        for dim_id, name in zip(dim_ids, names):
            if dim_id[0] not in {1, 2, 3}:
                raise ValueError("Dimension must be 1, 2, or 3")
            self.data[dim_id] = name
    
    def update_entries(self, parent_entries, child_entries):
        """
        Updates the tracker based on transformations.
        :param parent_entries: List of tuples [(orig_dim, orig_id), ...]
        :param child_entries: List of lists of tuples [[(new_dim. new_id)],[(new_dim, new_id), ...]...]
        Parent and child entries must be mapped one old to one new entry
        """
        if len(parent_entries) != len(child_entries): 
            raise ValueError("Original entries and new entries lists must be of the same length.")
        
        
        for orig, children in zip(parent_entries, child_entries):
            print("Orig: ",orig," Children: ",children)
            if orig not in self.data:
                    raise KeyError(f"Original entry {orig} not found.")
            parent_name = self.data[orig]
            print("Children: ",children)
            if children == []: # this conditional may be soon depreciated, only remains for corner case
                remove_name = "remove"
                self.data[orig] = remove_name   
            else:
                for new in children:
                    print(f"Adding {new} : {parent_name}")
                    #parent_name = self.data[orig]
                    self.data[new] = parent_name  # Inherit the name
                    self.parent_child_map[orig].append(new)  # Track the split
    
    def get_hierarchy(self):
        """
        Returns the parent-child hierarchy as a dictionary.
        """
        return dict(self.parent_child_map)
    
    def lookup_name(self, dim_id):
        """
        Looks up the name associated with a given dimension ID.
        :param dim_id: Tuple (dimension, id)
        :return: Name string if found, otherwise None
        """
        return self.data.get(dim_id, None)
    
    def get_name_counts(self, by_dimension=False):
        """
        Returns a count of each unique name in the dictionary.
        If by_dimension is True, returns a nested dictionary with counts per dimension.
        """
        if by_dimension:
            name_counts = defaultdict(lambda: defaultdict(int))
            for (dim, _), name in self.data.items():
                name_counts[dim][name] += 1
            return {dim: dict(names) for dim, names in name_counts.items()}
        else:
            name_counts = defaultdict(int)
            for name in self.data.values():
                name_counts[name] += 1
            return dict(name_counts)
    def get_sorted_by_prefix(self):
        """
        Returns a list of lists where elements are grouped and sorted by name prefix.
        """
        prefix_groups = defaultdict(list)
        
        for dim_id, name in self.data.items():
            prefix = name.split()[0] if " " in name else name  # Extract prefix
            prefix_groups[prefix].append((dim_id, name))
        
        return [sorted(group, key=lambda x: x[1]) for group in prefix_groups.values()]
        
    
    def __repr__(self):
        return f"Data: {self.data}\nParent-Child Map: {dict(self.parent_child_map)}"

In [6]:
def sort_by_prefix(prefixes, strings):
    """
    Groups strings into lists based on their prefixes.

    Parameters:
    prefixes (list of str): List of prefixes.
    strings (list of str): List of strings to be sorted.

    Returns:
    list of lists: A list where each sublist contains strings matching a prefix.
    """
    grouped = defaultdict(list)

    for s in strings:
        for prefix in prefixes:
            if s.startswith(prefix):
                grouped[prefix].append(s)
                break  # Stop checking once a match is found

    return [grouped[prefix] for prefix in prefixes]

In [9]:
######## THIS IS WORKING WITH THE EXCEPTION OF THE OUTER BOXES ########### replicating to get an idea about how to continue 



boundaries = (196000,219000,374000,388000,-3000, 33)
xmin, xmax, ymin, ymax, zmin, zmax = boundaries
extents = (xmin, (xmax-xmin), ymin, (ymax-ymin), zmin, (zmax-zmin))

gmsh.initialize()
gmsh.model.add("divided_volume")

    # Define a rectangular box in Gmsh using OCC kernel
    #length, width, height = box_dimensions
    #box = gmsh.model.occ.add_box(0, 0, 0, length, width, height)

box = gmsh.model.occ.add_box(extents[0], extents[2], extents[4], extents[1], extents[3], extents[5])
    # Synchronize after adding the box
face_names =  ['West', 'East', 'South', 'North', 'Base', 'Top'] #box surfaces creation order (ZY- , ZY+ , ZX- , ZX+ , XY- , XY+) or (W, E, S, N, Dn, Up)
tracker = DimensionIDTracker()
gmsh.model.occ.synchronize()


box_2dim = gmsh.model.occ.get_entities(dim=2)

tracker.add_entry(box_2dim, face_names)

"""
vert_surface = pv.Plane(center=(0.5,0.5,0.5), direction=(0,1,0), i_size=1.0, j_size=1.0, i_resolution=2, j_resolution=2)
vert2_surface = pv.Plane(center=(0.5,0.2,0.5), direction=(0,1,0), i_size=1.0, j_size=1.0, i_resolution=2, j_resolution=2)
horiz_surface = pv.Plane(center=(0.5,0.5,0.3), direction=(0,0,1), i_size=1.0, j_size=1.0, i_resolution=2, j_resolution=2)
vert_surf = msh.clip_polydata(vert_surface, xmin, xmax, ymin, ymax, zmin, zmax)
vert2_surf = msh.clip_polydata(vert2_surface, xmin, xmax, ymin, ymax, zmin, zmax) 
horiz_surf = msh.clip_polydata(horiz_surface, xmin, xmax, ymin, ymax, zmin, zmax)

all_surfaces = [vert_surf, vert2_surf, horiz_surf]
names = ['Vertical', 'Vertical2', 'Horizontal']
volume_names = ['Vol1','Vol2','Vol3','Vol4', 'Vol5', 'Vol6']
"""


surfaces_path = '/Users/amjonz/SIEGFRIED/Californie_fault_stability_analysis/CAL_mod8_epochD/model_surfaces/'

zeeland = pv.read(f'{surfaces_path}Zeeland_cal_mod8_d.vtk').delaunay_2d()
namurian = pv.read(f'{surfaces_path}namurian_cal_mod8_d.vtk').delaunay_2d()
nsg = pv.read(f'{surfaces_path}nsg_cal_mod8_d.vtk').delaunay_2d()
tegel = pv.read(f'{surfaces_path}tegel_cal_mod8_d.vtk').delaunay_2d()
dulk =  pv.read(f'{surfaces_path}dulk_cal_mod8_d.vtk').delaunay_2d()
belf =  pv.read(f'{surfaces_path}belf_cal_mod8_d.vtk').delaunay_2d()
vier = pv.read(f'{surfaces_path}vier_cal_mod8_d.vtk').delaunay_2d()
    #print(type(tegel))
#print(f"Currently building mesh from surfaces: {model_iter}_{model_suffix:04d}")
xmin, xmax, ymin, ymax, zmin, zmax = boundaries

clip_zeeland = msh.clip_polydata(zeeland, xmin, xmax, ymin, ymax, zmin, zmax).clean()
clip_namurian = msh.clip_polydata(namurian, xmin, xmax, ymin, ymax, zmin, zmax).clean()
clip_NSG = msh.clip_polydata(nsg, xmin, xmax, ymin, ymax, zmin, zmax).clean()
clip_tegel = msh.clip_polydata(tegel, xmin, xmax, ymin, ymax, zmin, zmax).clean()
clip_dulky = msh.clip_polydata(dulk, xmin, xmax, ymin, ymax, zmin, zmax).clean()
clip_belfy = msh.clip_polydata(belf, xmin, xmax, ymin, ymax, zmin, zmax).clean()
clip_vier = msh.clip_polydata(vier, xmin, xmax, ymin, ymax, zmin, zmax).clean()

all_surfaces = [clip_zeeland,clip_namurian,clip_NSG,clip_belfy,clip_tegel,clip_dulky,clip_vier]

volume_names = ['zeeland', 'namurian', 'NSG', 'belf', 'tegel', 'dulk', 'viersen']
all_dim3_prefix = volume_names

names = ['zeeland', 'namurian', 'NSG', 'belf', 'tegel', 'dulk', 'viersen']
#sides = ['Top', 'Base', 'East', 'West', 'North', 'South']
all_dim2_prefix = names+face_names

for surface, name, vol_name in zip(all_surfaces, names, volume_names):
    # input for surface adding loop: surfaces (pv.Polydata), names (list of str)
    points = surface.points
    faces = surface.faces.reshape((-1, 4))[:, 1:]
    #points.round(2), faces 

    gmsh_points = []
    #points = [(0,0,.5), (0,1,.5), (1,1,.5), (1,0,.5)]
    for pt in points:
        gmsh_points.append(gmsh.model.occ.add_point(pt[0], pt[1], pt[2]))
    print("Points ",gmsh_points)

    gmsh_curves = []
    for face in faces:
            lines = []
            for j in range(len(face)):
                p1 = gmsh_points[face[j]]
                p2 = gmsh_points[face[(j + 1) % len(face)]]
                lines.append(gmsh.model.occ.add_line(p1, p2))
                print(f"line {lines} {j}")
            loop = gmsh.model.occ.add_curve_loop(lines)
            print("loop ", loop)
            surface_tag = gmsh.model.occ.add_plane_surface([loop])
            print("adding surface tag : ", surface_tag)
            gmsh_curves.append(surface_tag)

    print("Curves :", gmsh_curves)
    
    namer = [name] * len(gmsh_curves)
    print(namer)
    #adding surface tags and names to the name tracker 
    tracker.add_entry([(2, plane_parts) for plane_parts in gmsh_curves], namer)
   

    gmsh.model.occ.synchronize()


    #print("Box tags without dimension: ",boxtags)
    vol = gmsh.model.occ.get_entities(dim=3) #starts with original the original box.
    all_surf = gmsh.model.occ.get_entities(dim=2)

    #Checking the current surfaces
    print("All surface tags: ", all_surf)
    print(all_surf)

    #Checking the current volumes
    #print("vol length ", len(vol), vol)

    vol_namer = [vol_name] * len(vol)
    print("Vol Name : ", vol_name)
    print("Vol : ", vol)

    #Adding Names to the volumes in the name tracker, ** ToDo **  this is clumsy and has no logic yet..
    tracker.add_entry(vol, vol_name)
    print("gmsh _curves ",gmsh_curves)
  
    box_surfaces = all_surf+vol

    ovv, ov = gmsh.model.occ.fragment(box_surfaces, all_surf, removeTool=True, removeObject=True) #[(2, plane_parts) for plane_parts in gmsh_curves]
    gmsh.model.occ.synchronize()
    for e in zip(box_surfaces, ov):
        print("parent " + str(e[0]) + " -> child " + str(e[1]))
    print("ov = ",len(ov),ov)
    print("ovv = ",len(ovv),ovv)

    print("Box Surfaces = ", len(box_surfaces), box_surfaces)
    tracker.update_entries(box_surfaces+all_surf, ov)

counts = tracker.get_name_counts(by_dimension=True)
#counts[2] holds the surface counts for the 2D surfaces with a certain name
#counts[3] holds the volumes counts for the number of 3D volumes


#grab all the 2D and 3D tags for the mesh
tags_3dim = gmsh.model.occ.get_entities(dim=3)
tags_2dim = gmsh.model.occ.get_entities(dim=2) 


#print('Dim3 Tags: ', tags_3dim)
dim3_names_list = []
for tag in tags_3dim:
     print('current tag 3', tag)
     gmsh.model.add_physical_group(3, [tag[1]], tag=tag[1])
     gmsh.model.set_physical_name(3, tag[1], f"{tracker.lookup_name(tag)}_{tag[1]:04d}")
     dim3_names_list.append(f"{tracker.lookup_name(tag)}_{tag[1]:04d}")
     counts[3][tracker.lookup_name(tag)] = counts[3][tracker.lookup_name(tag)]-1 #volumes physical group name is numbered by the remaining volumes that name, should count back to zero
#print("Dim 2 frag tags: ",tags_2dim)
dim2_names_list = []
for tag in tags_2dim:
     print('current tag 2', tag)
     gmsh.model.add_physical_group(2, [tag[1]], tag=tag[1])
     #gmsh.model.set_physical_name(2, tag[1], f"{tracker.lookup_name(tag)}_{counts[2][tracker.lookup_name(tag)]+10000}")
     gmsh.model.set_physical_name(2, tag[1], f"{tracker.lookup_name(tag)}_{counts[2][tracker.lookup_name(tag)]:04d}")
     dim2_names_list.append(f"{tracker.lookup_name(tag)}_{counts[2][tracker.lookup_name(tag)]:04d}")
     counts[2][tracker.lookup_name(tag)] = counts[2][tracker.lookup_name(tag)]-1 #surfaces physical group name is numbered by the remaining surfaces with that name, should count back to zero

dim2_sorted_surfaces = sort_by_prefix(all_dim2_prefix, dim2_names_list)
dim3_sorted_surfaces = sort_by_prefix(all_dim3_prefix, dim3_names_list)
print(counts)

gmsh.model.occ.synchronize()
pnt_entities = gmsh.model.occ.get_entities(dim=0)
gmsh.model.mesh.setSize(pnt_entities, 200)
#gmsh.option.setNumber("Mesh.MeshSizeMin", .3)
gmsh.model.mesh.generate(3)
gmsh.write('../test_results/califorie_test.msh')
gmsh.finalize()

#return dim2_sorted_surfaces, dim3_sorted_surfaces


######## THIS IS REPLICATED BELOW TO TRY SOMETHING ELSE ####### 

Points  [9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225

ValueError: Dimension ID list and names list must be of the same length.

In [5]:
counter = tracker.get_name_counts(by_dimension=True)

tag, counter[2][tracker.lookup_name(tag)]

((2, 7265), 39)

In [6]:
tracker.get_sorted_by_prefix()

[[((2, 1), 'West'),
  ((2, 850), 'West'),
  ((2, 854), 'West'),
  ((2, 1730), 'West'),
  ((2, 1734), 'West'),
  ((2, 2561), 'West'),
  ((2, 2604), 'West'),
  ((2, 3192), 'West'),
  ((2, 3280), 'West'),
  ((2, 3319), 'West'),
  ((2, 3449), 'West'),
  ((2, 3487), 'West'),
  ((2, 3617), 'West'),
  ((2, 3651), 'West'),
  ((2, 3707), 'West')],
 [((2, 2), 'East'),
  ((2, 853), 'East'),
  ((2, 857), 'East'),
  ((2, 1733), 'East'),
  ((2, 1737), 'East'),
  ((2, 2438), 'East'),
  ((2, 2519), 'East'),
  ((2, 2564), 'East'),
  ((2, 2607), 'East')],
 [((2, 3), 'South'),
  ((2, 851), 'South'),
  ((2, 855), 'South'),
  ((2, 1731), 'South'),
  ((2, 1735), 'South'),
  ((2, 2562), 'South'),
  ((2, 2605), 'South'),
  ((2, 3285), 'South'),
  ((2, 3193), 'South'),
  ((2, 3453), 'South'),
  ((2, 3320), 'South'),
  ((2, 3632), 'South'),
  ((2, 3488), 'South'),
  ((2, 3709), 'South'),
  ((2, 3652), 'South'),
  ((2, 4466), 'South'),
  ((2, 4375), 'South'),
  ((2, 4648), 'South'),
  ((2, 4629), 'South'),
  ((2

In [7]:
names = ['zeeland', 'namurian', 'NSG', 'belf', 'tegel', 'dulk', 'viersen']
sides = ['Top', 'Base', 'East', 'West', 'North', 'South']
all_names = names+sides

sorted = sort_by_prefix(all_names, dim2_names_list)
sorted[12]

['South_0039',
 'South_0038',
 'South_0037',
 'South_0036',
 'South_0035',
 'South_0034',
 'South_0033',
 'South_0032',
 'South_0031',
 'South_0030',
 'South_0029',
 'South_0028',
 'South_0027',
 'South_0026',
 'South_0025',
 'South_0024',
 'South_0023',
 'South_0022',
 'South_0021',
 'South_0020']

In [42]:
import mesher as mesh

all_surfaces = [clip_zeeland,clip_namurian,clip_NSG,clip_belfy,clip_tegel,clip_dulky,clip_vier]

volume_names = ['zeeland', 'namurian', 'NSG', 'belf', 'tegel', 'dulk', 'viersen']
all_dim3_prefix = volume_names

names = ['zeeland', 'namurian', 'NSG', 'belf', 'tegel', 'dulk', 'viersen']
#sides = ['Top', 'Base', 'East', 'West', 'North', 'South']
all_dim2_prefix = names+face_names


dim2, dim3 = mesh.build_mesh_from_surfaces(all_surfaces=all_surfaces, surface_names=names, volume_names=volume_names, model_name="test_accounting", output_file='./mesher.msh', bounding_box=boundaries, fieldsize=500)

TypeError: build_mesh_from_surfaces() got an unexpected keyword argument 'output_file'